# Silver Customer Events

Clean and transform customer events data from bronze to silver layer with incremental loading, data quality checks (clean, validate keys, dedup), and MERGE upserts via the common SilverLoader.

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/CommonHelper"

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/SilverLoader"

In [0]:
silver_table_name = get_table_name("customer_events", "silver")
bronze_table_name = get_table_name("customer_events", "bronze")

In [0]:
# spark.read.table(bronze_table_name).display()

In [0]:
from pyspark.sql import functions as F

# --- Transform function for silver customer events ---
# Must carry _bronze_ingestion_ts through (required by SilverLoader for watermark tracking)
def transform_customer_events(bronze_df):
    silver_df = (bronze_df
        .select(
            F.col("event_id"),
            F.col("customer_id"),
            clean_text("event_type").alias("event_type"),
            F.col("event_timestamp"),
            clean_text("device_type").alias("device_type"),
            clean_text("page").alias("page"),
            clean_text("metadata").alias("metadata"),
            F.col("_bronze_ingestion_ts")
        )
    )
    # Validate keys — drop rows with null/empty/non-positive event_id
    silver_df = validate_keys(silver_df, ["event_id"])
    # Remove duplicates by event_id, keeping the latest event timestamp
    silver_df = remove_duplicates(silver_df, ["event_id"], order_col="event_timestamp")
    # Drop any remaining bronze metadata columns
    silver_df = drop_bronze_metadata(silver_df)
    return silver_df

# --- Run incremental load (full on first run, MERGE on subsequent runs) ---
result = incremental_silver_load(
    bronze_table_name=bronze_table_name,
    silver_table_name=silver_table_name,
    key_columns=["event_id"],
    transform_fn=transform_customer_events,
    update_condition="source.event_timestamp >= target.event_timestamp"
)

print(result)

In [0]:
display(spark.read.table(silver_table_name))